# DVF 2022 — Le manquant suit le type de bien

Ce notebook examine si l'absence de certaines valeurs dépend du type de bien de la ligne. Il croise le type de local avec la présence de la surface bâtie et du nombre de pièces. L'objectif est d'observer si le manquant est aléatoire ou lié à la nature du bien.

## Cellule 1 — Connexion au fichier

Interrogation avec DuckDB, directement au format Parquet. Le chemin pointe vers le fichier `dvf-2022.parquet` placé dans le dossier `data/`.

In [1]:
import duckdb
from pathlib import Path

# Chemin vers le fichier Parquet, place dans le dossier data/
FICHIER = Path(r"./data/dvf-2022.parquet")

con = duckdb.connect()
pq = str(FICHIER)
assert FICHIER.exists(), f"Fichier introuvable : {FICHIER}"
nb_lignes = con.execute(f"SELECT count(*) FROM '{pq}'").fetchone()[0]
print(f"Fichier : {FICHIER.name}")
print(f"Lignes  : {nb_lignes:,}".replace(',', ' '))

Fichier : dvf-2022.parquet
Lignes  : 4 617 590


## Cellule 2 — Présence de la surface bâtie et du nombre de pièces, par type de local

Pour chaque type de local (les lignes sans local sont regroupées sous « (terrain / sans local) »), on compte le nombre de lignes, puis le nombre et la part de lignes qui portent une surface bâtie et un nombre de pièces. Une valeur à 0 (cas des dépendances) est comptée comme présente ; seule l'absence (valeur vide) est comptée comme manquante.

In [2]:
manquant_par_type = con.execute(f"""
    SELECT COALESCE("Type local", '(terrain / sans local)') AS type_local,
           count(*)                              AS nb_lignes,
           count("Surface reelle bati")          AS avec_surface_batie,
           ROUND(100.0 * count("Surface reelle bati") / count(*), 1) AS pct_surface,
           count("Nombre pieces principales")    AS avec_nb_pieces,
           ROUND(100.0 * count("Nombre pieces principales") / count(*), 1) AS pct_pieces
    FROM '{pq}'
    GROUP BY 1
    ORDER BY nb_lignes DESC
""").fetchdf()

manquant_par_type

,type_local,nb_lignes,avec_surface_batie,pct_surface,avec_nb_pieces,pct_pieces
0,(terrain / sans local),1876728,0,0.0,0,0.0
1,Dépendance,1203439,1203416,100.0,1203416,100.0
2,Maison,756009,755946,100.0,755946,100.0
3,Appartement,638879,638855,100.0,638855,100.0
4,Local industriel. commercial ou assimilé,142535,139991,98.2,139991,98.2
